In [1]:
!pip install sqlalchemy pymysql pandas

In [2]:
import pandas as pd
from sqlalchemy import create_engine, URL
from getpass import getpass

password = getpass("password")



password ········


In [3]:
connection_url = URL.create(
    "mysql+pymysql",
    username="Kumari",
    password=password,
    host="localhost",
    port=3307,
    database="giftora"
)

engine = create_engine(connection_url)

In [4]:
with engine.connect() as conn:
    print("MySQL connection successful!")

MySQL connection successful!


In [5]:
df = pd.read_sql(
    "SELECT * FROM online_retail_clean",
    engine
)

print("Shape:", df.shape)
df.head()

Shape: (1033036, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom


In [6]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'CustomerID', 'Country'],
      dtype='object')

In [7]:
df.sample(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
539825,540977,21222,SET/4 BADGES BEETLES,3,2011-01-12 15:01:00,0.85,NaN,United Kingdom
387519,527754,21977,PACK OF 60 PINK PAISLEY CAKE CASES,5,2010-10-18 17:05:00,0.55,15710.0,United Kingdom
23188,491394,21477,STRING OF 20 ROSE LIGHTS,1,2009-12-10 16:21:00,12.75,15916.0,United Kingdom
114537,500690,81985A,None,-10,2010-03-09 12:35:00,0.00,NaN,United Kingdom
818039,566617,22182,CAKE STAND VICTORIAN FILIGREE SMALL,12,2011-09-14 09:11:00,4.95,18092.0,United Kingdom
94051,498545,85198,ASSORTED FARMYARD ANIMALS IN BUCKET,12,2010-02-19 17:02:00,0.85,13191.0,United Kingdom
147807,503905,22276,WASH BAG VINTAGE ROSE PAISLEY,1,2010-04-08 12:23:00,2.55,17961.0,United Kingdom
992635,580260,85123A,WHITE HANGING HEART T-LIGHT HOLDER,3,2011-12-02 12:36:00,2.95,14085.0,United Kingdom
22652,491204,22044,CHRISTMAS CARD STACK OF PRESENTS,12,2009-12-10 13:59:00,0.42,14648.0,United Kingdom
1019048,C512274,21313,GLASS HEART T-LIGHT HOLDER,-1,2010-06-14 13:35:00,0.85,14615.0,United Kingdom


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   CustomerID   797885 non-null   float64       
 7   Country      1033036 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 63.1+ MB


In [9]:
df.describe()

,Quantity,InvoiceDate,Price,CustomerID
count,1.033036e+06,1033036,1.033036e+06,797885.000000
mean,1.007688e+01,2011-01-03 14:30:35.429549824,4.613980e+00,15313.062777
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-05 11:38:00,1.250000e+00,13964.000000
50%,3.000000e+00,2010-12-09 13:34:00,2.100000e+00,15228.000000
75%,1.000000e+01,2011-07-27 13:17:00,4.150000e+00,16788.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.751976e+02,NaN,1.223975e+02,1696.466663


In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df = df.astype({
    'Invoice': 'string',
    'StockCode': 'string',
    'Description': 'string',
    'CustomerID': 'Int64',
    'Country': 'string'
})

In [12]:
# Create a separate dataframe
rfm_data = df.dropna(subset=['CustomerID']).copy()

# Reset index
rfm_data.reset_index(drop=True, inplace=True)

# Verify
print("Original Dataset Shape:", df.shape)
print("RFM Dataset Shape:", rfm_data.shape)

print("Missing CustomerID:", rfm_data['CustomerID'].isnull().sum())

Original Dataset Shape: (1033036, 8)
RFM Dataset Shape: (797885, 8)
Missing CustomerID: 0


In [13]:
# Keep positive quantity transactions
rfm_data = rfm_data[rfm_data['Quantity'] > 0].copy()

# Remove cancelled invoices (invoice numbers starting with C)
rfm_data = rfm_data[
    ~rfm_data['Invoice'].str.startswith('C', na=False)
].copy()

print("Shape after filtering:", rfm_data.shape)

Shape after filtering: (779495, 8)


In [14]:
#total revenue or transaction amount
rfm_data['TotalAmount'] = (
    rfm_data['Quantity'] * rfm_data['Price']
)

rfm_data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalAmount
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,59.5
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,39.6


In [15]:
#Define snapshot date
snapshot_date = rfm_data['InvoiceDate'].max() + pd.Timedelta(days=1)

print("Snapshot Date:", snapshot_date)

Snapshot Date: 2011-12-10 12:50:00


In [16]:
rfm = rfm_data.groupby('CustomerID').agg({

    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,

    'Invoice': 'nunique',

    'TotalAmount': 'sum'

}).reset_index()

# Rename columns
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [17]:
#calculate RFM scores
# Recency: Lower value = Higher score
rfm['R_Score'] = pd.qcut(
    rfm['Recency'].rank(method='first', ascending=False),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Frequency: Higher value = Higher score
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


# Monetary: Higher value = Higher score
rfm['M_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [18]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346,326,12,77556.46,2,5,5
1,12347,2,8,4921.53,5,4,5
2,12348,75,5,2019.40,3,4,4
3,12349,19,4,4428.69,4,3,5
4,12350,310,1,334.40,2,1,2


In [19]:
# Combine individual scores
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

# Calculate total score
rfm['RFM_Total'] = (
    rfm['R_Score'] +
    rfm['F_Score'] +
    rfm['M_Score']
)

rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total
0,12346,326,12,77556.46,2,5,5,255,12
1,12347,2,8,4921.53,5,4,5,545,14
2,12348,75,5,2019.40,3,4,4,344,11
3,12349,19,4,4428.69,4,3,5,435,12
4,12350,310,1,334.40,2,1,2,212,5
5,12351,375,1,300.93,2,1,2,212,5
6,12352,36,10,2849.84,4,5,4,454,13
7,12353,204,2,406.76,2,2,2,222,6
8,12354,232,1,1079.40,2,1,3,213,6
9,12355,214,2,947.61,2,2,3,223,7


In [20]:
def segment_customer(row):

    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'

    elif row['R_Score'] >= 3 and row['F_Score'] >= 4:
        return 'Loyal Customers'

    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New / Promising Customers'

    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'

    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Lost Customers'

    else:
        return 'Potential / Other Customers'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['Ischurned']=rfm['Recency']>90

In [21]:
rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Total,Segment,Ischurned
0,12346,326,12,77556.46,2,5,5,255,12,At Risk,True
1,12347,2,8,4921.53,5,4,5,545,14,Champions,False
2,12348,75,5,2019.40,3,4,4,344,11,Loyal Customers,False
3,12349,19,4,4428.69,4,3,5,435,12,Potential / Other Customers,False
4,12350,310,1,334.40,2,1,2,212,5,Lost Customers,True
5,12351,375,1,300.93,2,1,2,212,5,Lost Customers,True
6,12352,36,10,2849.84,4,5,4,454,13,Champions,False
7,12353,204,2,406.76,2,2,2,222,6,Lost Customers,True
8,12354,232,1,1079.40,2,1,3,213,6,Lost Customers,True
9,12355,214,2,947.61,2,2,3,223,7,Lost Customers,True


In [22]:
# MERGE RFM RESULTS BACK ONTO TRANSACTIONS AND EXPORT

df = df[df['CustomerID'].notna() & (df['Quantity'] > 0)].copy()

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'Ischurned']],
    on='CustomerID', how='left'
)
df_final.head()



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,Ischurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-12-01,30.0,158,8,2433.28,3,4,4,Loyal Customers,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,2009-12-01,59.5,158,8,2433.28,3,4,4,Loyal Customers,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-12-01,30.0,158,8,2433.28,3,4,4,Loyal Customers,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-12-01,100.8,158,8,2433.28,3,4,4,Loyal Customers,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,2009-12-01,39.6,158,8,2433.28,3,4,4,Loyal Customers,True


In [23]:
df_final.to_excel('EcommerceDashboardData.xlsx', index=False)